# 01 Fetch Data
This notebook fetches the datasets required from the web.

## Set up and configuration

In [27]:
# download required datasets
from urllib.request import urlretrieve # library for requesting data
import os

In [28]:
# from the current `tute_1` directory, go back two levels to the root of the directory
data_dir = '../data/raw/'

# check if it exists as makedir will raise an error if it does exist
if not os.path.exists(data_dir):
    os.makedirs(data_dir)
    
# now, for each type of data set we will need, we will create the paths
for target_dir in ('tlc_trips', 'mta_ridership', 'mta_alerts', 'meteostat_weather'): 
    if not os.path.exists(data_dir + target_dir):
        os.makedirs(data_dir + target_dir)

In [29]:
# specify time range of data
YEARS = [2025, 2026]
MONTHS = range(1, 13)

## Fetch TLC Trip Record Data

In [30]:
# set required data URL for tlc data
URL_ROOT_YELLOW = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_"#year-month.parquet
URL_ROOT_GREEN = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_"#year-month.parquet
URL_ROOT_FHVHV = "https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_"#year-month.parquet

URL_ROOT = [URL_ROOT_YELLOW, URL_ROOT_GREEN, URL_ROOT_FHVHV]
TAXI_TYPES = ["yellow", "green", "fhvhv"]

In [31]:
# data output directory is `data/tlc_trips/`
tlc_output_dir = data_dir + 'tlc_trips/'

for taxi_type, url_root in zip(TAXI_TYPES, URL_ROOT):
    for year in YEARS:
        for month in MONTHS:
            if (year==2026) and (month>=6): # data available only up to 2026 May
                continue
            
            if os.path.exists(f"{tlc_output_dir}/{taxi_type}/{year}-{month:02d}.parquet"): # temp here to skip already downloaded files
                print(f"Skipping  {taxi_type} {year}/{month:02d}")
                continue

            # generate url
            url = f'{url_root}{year}-{month:02d}.parquet'
            # generate output location and filename
            output_dir = f"{tlc_output_dir}/{taxi_type}/{year}-{month:02d}.parquet"
            # download
            urlretrieve(url, output_dir) 
            
            print(f"Completed {taxi_type} {year}/{month:02d}")

Skipping  yellow 2025/01
Skipping  yellow 2025/02
Skipping  yellow 2025/03
Completed yellow 2025/04
Skipping  yellow 2025/05
Skipping  yellow 2025/06
Skipping  yellow 2025/07
Skipping  yellow 2025/08
Skipping  yellow 2025/09
Skipping  yellow 2025/10
Skipping  yellow 2025/11
Skipping  yellow 2025/12
Skipping  yellow 2026/01
Skipping  yellow 2026/02
Skipping  yellow 2026/03
Skipping  yellow 2026/04
Skipping  yellow 2026/05
Skipping  green 2025/01
Skipping  green 2025/02
Skipping  green 2025/03
Skipping  green 2025/04
Skipping  green 2025/05
Skipping  green 2025/06
Skipping  green 2025/07
Skipping  green 2025/08
Skipping  green 2025/09
Skipping  green 2025/10
Skipping  green 2025/11
Skipping  green 2025/12
Skipping  green 2026/01
Skipping  green 2026/02
Skipping  green 2026/03
Skipping  green 2026/04
Skipping  green 2026/05
Skipping  fhvhv 2025/01
Skipping  fhvhv 2025/02
Skipping  fhvhv 2025/03
Skipping  fhvhv 2025/04
Skipping  fhvhv 2025/05
Skipping  fhvhv 2025/06
Skipping  fhvhv 2025/07

## Fetch MTA Subway Hourly Ridership Data
Only select where transit_mode = subway (exclude Staten Island Railway and tram). 

In [ ]:
import requests
from pathlib import Path

def download_mta_ridership_month(year, month, output_dir):
    """
    Downloads MTA Subway Hourly Ridership Data per month after querying the API to group by (station, hourly timestamp) and aggregated by `ridership`. 
    Raw fare, payment categories are collapsed. Saves one month of data at once to given `output_dir`.
    """

    start = f"{year}-{month:02d}-01T00:00:00"

    if month == 12:
        end = f"{year + 1}-01-01T00:00:00"
    else:
        end = f"{year}-{month + 1:02d}-01T00:00:00"

    url = "https://data.ny.gov/resource/5wq4-mkjj.csv" # SODA2 API endpoint

    # Query API to group and aggregate on server-side to reduce download size
    params = {
        "$select": """
            transit_timestamp,
            station_complex_id,
            station_complex,
            borough,
            latitude,
            longitude,
            sum(ridership) as ridership,
            sum(transfers) as transfers
        """,
        "$where": f"""
            transit_mode = 'subway'
            AND transit_timestamp >= '{start}'
            AND transit_timestamp < '{end}'
        """,
        "$group": """
            transit_timestamp,
            station_complex_id,
            station_complex,
            borough,
            latitude,
            longitude
        """,
        "$order": "transit_timestamp, station_complex_id",
        "$limit": 1000000
    }

    # Create directory for output path if doesn't already exist
    output_path = Path(output_dir) / f"{year}-{month:02d}.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Check if already downloaded
    if output_path.exists():
        print(f"Skipping MTA ridership {year}/{month:02d}")
        return
    
    # Request data from API
    response = requests.get(url, params=params, timeout=120)
    response.raise_for_status()

    # Save API response to local CSV
    output_path.write_bytes(response.content)
    print(f"Saved {output_path}")

In [ ]:
# Set mta_ridership output directory
mta_ridership_output_dir = data_dir + 'mta_ridership/'

# Call specified download function for 2025 Jan - 2026 May
for year in YEARS:
    for month in MONTHS:
        if (year==2026) and (month>=6): # data available only up to 2026 May
            continue
        download_mta_ridership_month(year, month, mta_ridership_output_dir)

Saved ../data/raw/mta_ridership/2025-01.csv
Saved ../data/raw/mta_ridership/2025-02.csv
Saved ../data/raw/mta_ridership/2025-03.csv
Saved ../data/raw/mta_ridership/2025-04.csv
Saved ../data/raw/mta_ridership/2025-05.csv
Saved ../data/raw/mta_ridership/2025-06.csv
Saved ../data/raw/mta_ridership/2025-07.csv
Saved ../data/raw/mta_ridership/2025-08.csv
Saved ../data/raw/mta_ridership/2025-09.csv
Saved ../data/raw/mta_ridership/2025-10.csv
Saved ../data/raw/mta_ridership/2025-11.csv
Saved ../data/raw/mta_ridership/2025-12.csv
Saved ../data/raw/mta_ridership/2026-01.csv
Saved ../data/raw/mta_ridership/2026-02.csv
Saved ../data/raw/mta_ridership/2026-03.csv
Saved ../data/raw/mta_ridership/2026-04.csv
Saved ../data/raw/mta_ridership/2026-05.csv


## Fetch MTA Service Alerts Data
Only select where agency = NYCT Subway (to include only subway related alerts). 

In [18]:
def download_mta_alerts_month(year, month, output_dir):
    """
    Downloads MTA Service Alerts Data per month after querying the API to filter for certain attributes listed below. 
    Only alerts related to subway services are downloaded. Saves one month of data at once to given `output_dir`.
    """

    start = f"{year}-{month:02d}-01T00:00:00"

    if month == 12:
        end = f"{year + 1}-01-01T00:00:00"
    else:
        end = f"{year}-{month + 1:02d}-01T00:00:00"

    url = "https://data.ny.gov/resource/7kct-peq7.csv" # SODA2 API endpoint

    # Query API to filter and select on server-side to reduce download size
    params = {
        "$select": """
            alert_id,
            event_id,
            update_number,
            date,
            agency,
            status_label,
            affected,
            header,
            description
        """,
        "$where": f"""
            agency = 'NYCT Subway'
            AND date >= '{start}'
            AND date < '{end}'
        """,
        "$order": "date, event_id, update_number",
        "$limit": 100000
    }

    # Create directory for output path if doesn't already exist
    output_path = Path(output_dir) / f"{year}-{month:02d}.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Check if already downloaded
    if output_path.exists():
        print(f"Skipping MTA alerts {year}/{month:02d}")
        return

    # Request data from API
    response = requests.get(url, params=params, timeout=120)
    response.raise_for_status()

    # Save API response to local CSV
    output_path.write_bytes(response.content)
    print(f"Saved {output_path}")

In [ ]:
# Set mta_alerts output directory
mta_alerts_output_dir = data_dir + 'mta_alerts/'

# Call specified download function for 2025 Jan - 2026 May
for year in YEARS:
    for month in MONTHS:
        if (year==2026) and (month>=6): # data available only up to 2026 May
            continue
        download_mta_alerts_month(year, month, mta_alerts_output_dir)

Saved ../data/raw/mta_alerts/2025-01.csv
Saved ../data/raw/mta_alerts/2025-02.csv
Saved ../data/raw/mta_alerts/2025-03.csv
Saved ../data/raw/mta_alerts/2025-04.csv
Saved ../data/raw/mta_alerts/2025-05.csv
Saved ../data/raw/mta_alerts/2025-06.csv
Saved ../data/raw/mta_alerts/2025-07.csv
Saved ../data/raw/mta_alerts/2025-08.csv
Saved ../data/raw/mta_alerts/2025-09.csv
Saved ../data/raw/mta_alerts/2025-10.csv
Saved ../data/raw/mta_alerts/2025-11.csv
Saved ../data/raw/mta_alerts/2025-12.csv
Saved ../data/raw/mta_alerts/2026-01.csv
Saved ../data/raw/mta_alerts/2026-02.csv
Saved ../data/raw/mta_alerts/2026-03.csv
Saved ../data/raw/mta_alerts/2026-04.csv
Saved ../data/raw/mta_alerts/2026-05.csv


## Fetch Meteostat Weather Data

In [ ]:
from datetime import datetime, timedelta
from meteostat import Point, Hourly
import calendar

def download_meteostat_weather_month(year, month, output_dir):
    """
    Downloads Meteostat Weather Data for New York City via the Meteostat API, filtering for the appropriate time frame.
    The weather in New York is treated as one point and the hourly weather data is saved as one CSV per month to given `output_dir`.
    """

    # Set start and end times of the month
    last_day = calendar.monthrange(year, month)[1]
    start = datetime(year, month, 1)
    end = datetime(year, month, last_day, 23, 59)

    # Representative NYC point; roughly Lower Manhattan / city centre
    nyc = Point(40.7128, -74.0060)

    # Create output path
    output_path = Path(output_dir) / f"{year}-{month:02d}.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Skip if already downloaded
    if output_path.exists():
        print(f"Skipping Meteostat Weather {year}/{month:02d}")
        return

    # Fetch data from Meteostat for each hour of specified month in NYC
    data = Hourly(nyc, start, end, timezone="America/New_York").fetch()
    weather_df = data.reset_index()

    # Save to local CSV
    weather_df.to_csv(output_path, index=False)
    print(f"Saved {output_path}")

In [24]:
# Set meteostat_weather output directory
meteostat_weather_output_dir = data_dir + "meteostat_weather/"

# Call specified download function for 2025 Jan - 2026 May
for year in YEARS:
    for month in MONTHS:
        if (year == 2026) and (month >= 6): # match other data time frame
            continue
        download_meteostat_weather_month(year, month, meteostat_weather_output_dir)

Saved ../data/raw/meteostat_weather/2025-01.csv
Saved ../data/raw/meteostat_weather/2025-02.csv
Saved ../data/raw/meteostat_weather/2025-03.csv
Saved ../data/raw/meteostat_weather/2025-04.csv
Saved ../data/raw/meteostat_weather/2025-05.csv
Saved ../data/raw/meteostat_weather/2025-06.csv
Saved ../data/raw/meteostat_weather/2025-07.csv
Saved ../data/raw/meteostat_weather/2025-08.csv
Saved ../data/raw/meteostat_weather/2025-09.csv
Saved ../data/raw/meteostat_weather/2025-10.csv
Saved ../data/raw/meteostat_weather/2025-11.csv
Saved ../data/raw/meteostat_weather/2025-12.csv
Saved ../data/raw/meteostat_weather/2026-01.csv
Saved ../data/raw/meteostat_weather/2026-02.csv
Saved ../data/raw/meteostat_weather/2026-03.csv
Saved ../data/raw/meteostat_weather/2026-04.csv
Saved ../data/raw/meteostat_weather/2026-05.csv


## Checkpoint
By the end of this Jupyter Notebook, all the relevant data from 2025 Jan - 2026 May (inclusive) should be downloaded in their respective folders:
- `tlc_trips` 
    - `yellow`
    - `green`
    - `fhvhv`
- `mta_ridership`
- `mta_alerts` 
- `meteostat_weather`

The datasets should be separated by month and cover the same time frame. Check using the `print_tree` function below.

In [ ]:
def print_tree(path, max_depth=3):
    """ 
    Print the directory structure for /data/raw/, which should contain tlc_trips/, mta_ridership/, mta_alerts/, and meteostat_weather/.
    """
    path = Path(path)
    base_depth = len(path.parts)

    for item in sorted(path.rglob("*")):
        depth = len(item.parts) - base_depth
        if depth > max_depth:
            continue

        indent = "    " * (depth - 1)
        prefix = "|-- "
        print(f"{indent}{prefix}{item.name}")

# Print the directory structure to check
print("data/raw/")
print_tree("../data/raw", max_depth=3)

data/raw/
|-- meteostat_weather
    |-- 2025-01.csv
    |-- 2025-02.csv
    |-- 2025-03.csv
    |-- 2025-04.csv
    |-- 2025-05.csv
    |-- 2025-06.csv
    |-- 2025-07.csv
    |-- 2025-08.csv
    |-- 2025-09.csv
    |-- 2025-10.csv
    |-- 2025-11.csv
    |-- 2025-12.csv
    |-- 2026-01.csv
    |-- 2026-02.csv
    |-- 2026-03.csv
    |-- 2026-04.csv
    |-- 2026-05.csv
|-- mta_alerts
    |-- 2025-01.csv
    |-- 2025-02.csv
    |-- 2025-03.csv
    |-- 2025-04.csv
    |-- 2025-05.csv
    |-- 2025-06.csv
    |-- 2025-07.csv
    |-- 2025-08.csv
    |-- 2025-09.csv
    |-- 2025-10.csv
    |-- 2025-11.csv
    |-- 2025-12.csv
    |-- 2026-01.csv
    |-- 2026-02.csv
    |-- 2026-03.csv
    |-- 2026-04.csv
    |-- 2026-05.csv
|-- mta_ridership
    |-- 2025-01.csv
    |-- 2025-02.csv
    |-- 2025-03.csv
    |-- 2025-04.csv
    |-- 2025-05.csv
    |-- 2025-06.csv
    |-- 2025-07.csv
    |-- 2025-08.csv
    |-- 2025-09.csv
    |-- 2025-10.csv
    |-- 2025-11.csv
    |-- 2025-12.csv
    |-- 2026-01